# 01 — Counter Availability and Data Quality

Run this notebook after `00_cycling_data_extraction.ipynb`.

It summarises:
- counter availability;
- observed and expected days;
- missing dates and continuous collection gaps;
- duplicate-looking raw records;
- coverage percentage.

Missing days are reported only; they are not automatically filled or treated as zero.

## 1. Repository configuration

In [1]:
import pandas as pd
import yaml
import os
from pathlib import Path


from pathlib import Path
import os
import yaml

def find_project_root(start=None):
    """Find the repository root by looking for .git or config.yaml."""
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists() or (candidate / "config.yaml").exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root()

config_path = PROJECT_ROOT / "config.yaml"
config = {}
if config_path.exists():
    with open(config_path, "r", encoding="utf-8") as f:
        config = yaml.safe_load(f) or {}

paths_cfg = config.get("paths", {})

RAW_ROOT = PROJECT_ROOT / paths_cfg.get("raw_dir", "data/raw")
PROCESSED_ROOT = PROJECT_ROOT / paths_cfg.get("processed_dir", "data/processed")

# Optional override for local machines:
# Windows PowerShell example:
#   $env:CYCLING_DATA_DIR="D:\\path\\to\\CYCLING_DATA"
CYCLING_DATA_DIR = Path(
    os.environ.get(
        "CYCLING_DATA_DIR",
        RAW_ROOT / "cycling_data"
    )
).expanduser().resolve()

CYCLING_PROCESSED_DIR = PROCESSED_ROOT / "cycling_counters"
CYCLING_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Cycling raw data:", CYCLING_DATA_DIR)
print("Cycling processed data:", CYCLING_PROCESSED_DIR)

EXTRACTION_DIR = CYCLING_PROCESSED_DIR / "extraction"
QUALITY_DIR = CYCLING_PROCESSED_DIR / "quality"
QUALITY_DIR.mkdir(parents=True, exist_ok=True)

DAILY_FILE = EXTRACTION_DIR / "daily_bicycle_volume.csv"
DUPLICATE_FILE = EXTRACTION_DIR / "duplicate_records.csv"

print("Daily file:", DAILY_FILE)
print("Duplicate file:", DUPLICATE_FILE)

Project root: C:\Users\f_tir\Documents\Code\Victoria-Urban-Planning
Cycling raw data: C:\Users\f_tir\OneDrive\Documents\UNI_2026\CAPSTONE PROJECT\A- TEAM A\CYCLING_DATA
Cycling processed data: C:\Users\f_tir\Documents\Code\Victoria-Urban-Planning\data\processed\cycling_counters
Daily file: C:\Users\f_tir\Documents\Code\Victoria-Urban-Planning\data\processed\cycling_counters\extraction\daily_bicycle_volume.csv
Duplicate file: C:\Users\f_tir\Documents\Code\Victoria-Urban-Planning\data\processed\cycling_counters\extraction\duplicate_records.csv


## 2. Load extraction outputs

In [2]:
daily = pd.read_csv(DAILY_FILE, parse_dates=["DATE_PARSED"])
duplicates = pd.read_csv(DUPLICATE_FILE, low_memory=False)

daily["SITE_ID"] = pd.to_numeric(daily["SITE_ID"], errors="coerce")
duplicates["SITE_ID"] = pd.to_numeric(duplicates["SITE_ID"], errors="coerce")

print("Daily rows:", len(daily))
print("Duplicate-report rows:", len(duplicates))

Daily rows: 3566
Duplicate-report rows: 39685


## 3. Define expected study periods

In [3]:
EXPECTED_PERIODS = {
    32493: {"street": "Wellington Street", "start": "2019-01-01", "end": "2022-12-31"},
    9077: {"street": "Albert Street", "start": "2019-01-01", "end": "2022-12-31"},
    34687: {"street": "Moorabool Street", "start": "2020-10-01", "end": "2022-12-31"},
    64644: {"street": "Moorabool linked ID", "start": "2020-10-01", "end": "2022-12-31"},
    64645: {"street": "Moorabool linked ID", "start": "2020-10-01", "end": "2022-12-31"},
    40004: {"street": "Heidelberg Road", "start": "2019-10-01", "end": "2022-03-31"},
    40005: {"street": "Heidelberg Road", "start": "2019-10-01", "end": "2022-03-31"},
}

## 4. Build availability and quality summaries

In [4]:
quality_rows = []
gap_rows = []
missing_rows = []

for site_id, meta in EXPECTED_PERIODS.items():
    site = daily.loc[daily["SITE_ID"] == site_id].copy()
    expected_start = pd.Timestamp(meta["start"])
    expected_end = pd.Timestamp(meta["end"])
    expected_dates = pd.date_range(expected_start, expected_end, freq="D")

    if site.empty:
        quality_rows.append({
            "SITE_ID": site_id,
            "STREET": meta["street"],
            "FOUND": "No",
            "FIRST_DATE": pd.NaT,
            "LAST_DATE": pd.NaT,
            "EXPECTED_DAYS": len(expected_dates),
            "OBSERVED_DAYS": 0,
            "MISSING_DATES": len(expected_dates),
            "COLLECTION_GAPS": 1 if len(expected_dates) else 0,
            "LONGEST_GAP_DAYS": len(expected_dates),
            "DUPLICATE_RECORDS": int((duplicates["SITE_ID"] == site_id).sum()),
            "COVERAGE_PCT": 0.0,
        })
        continue

    observed = pd.DatetimeIndex(site["DATE_PARSED"].dt.normalize().unique())
    missing = expected_dates.difference(observed)

    for d in missing:
        missing_rows.append({"SITE_ID": site_id, "STREET": meta["street"], "DATE": d})

    gaps = []
    if len(missing):
        start = prev = missing[0]
        for current in missing[1:]:
            if (current - prev).days > 1:
                gaps.append((start, prev))
                start = current
            prev = current
        gaps.append((start, prev))

    for start, end in gaps:
        gap_rows.append({
            "SITE_ID": site_id,
            "STREET": meta["street"],
            "GAP_START": start,
            "GAP_END": end,
            "GAP_DAYS": (end - start).days + 1,
        })

    quality_rows.append({
        "SITE_ID": site_id,
        "STREET": meta["street"],
        "FOUND": "Yes",
        "FIRST_DATE": site["DATE_PARSED"].min(),
        "LAST_DATE": site["DATE_PARSED"].max(),
        "EXPECTED_DAYS": len(expected_dates),
        "OBSERVED_DAYS": len(observed.intersection(expected_dates)),
        "MISSING_DATES": len(missing),
        "COLLECTION_GAPS": len(gaps),
        "LONGEST_GAP_DAYS": max(((e-s).days+1 for s,e in gaps), default=0),
        "DUPLICATE_RECORDS": int((duplicates["SITE_ID"] == site_id).sum()),
        "COVERAGE_PCT": round((len(observed.intersection(expected_dates)) / len(expected_dates))*100, 2),
    })

quality_report = pd.DataFrame(quality_rows)
collection_gaps = pd.DataFrame(gap_rows)
missing_dates = pd.DataFrame(missing_rows)

display(quality_report)
display(collection_gaps)

,SITE_ID,STREET,FOUND,FIRST_DATE,LAST_DATE,EXPECTED_DAYS,OBSERVED_DAYS,MISSING_DATES,COLLECTION_GAPS,LONGEST_GAP_DAYS,DUPLICATE_RECORDS,COVERAGE_PCT
0,32493,Wellington Street,Yes,2019-01-01,2022-12-25,1461,1455,6,1,6,9877,99.59
1,9077,Albert Street,Yes,2019-01-01,2022-12-25,1461,1287,174,2,168,1996,88.09
2,34687,Moorabool Street,No,NaT,NaT,822,0,822,1,822,0,0.00
3,64644,Moorabool linked ID,No,NaT,NaT,822,0,822,1,822,0,0.00
4,64645,Moorabool linked ID,No,NaT,NaT,822,0,822,1,822,0,0.00
5,40004,Heidelberg Road,Yes,2021-01-09,2022-03-31,913,412,501,5,466,13141,45.13
6,40005,Heidelberg Road,Yes,2021-01-09,2022-03-31,913,412,501,5,466,14671,45.13


,SITE_ID,STREET,GAP_START,GAP_END,GAP_DAYS
0,32493,Wellington Street,2022-12-26,2022-12-31,6
1,9077,Albert Street,2019-08-05,2020-01-19,168
2,9077,Albert Street,2022-12-26,2022-12-31,6
3,40004,Heidelberg Road,2019-10-01,2021-01-08,466
4,40004,Heidelberg Road,2021-12-13,2021-12-19,7
5,40004,Heidelberg Road,2022-01-03,2022-01-09,7
6,40004,Heidelberg Road,2022-02-07,2022-02-20,14
7,40004,Heidelberg Road,2022-02-28,2022-03-06,7
8,40005,Heidelberg Road,2019-10-01,2021-01-08,466
9,40005,Heidelberg Road,2021-12-13,2021-12-19,7


## 5. Export quality reports

In [5]:
quality_report.to_csv(QUALITY_DIR / "data_quality_report.csv", index=False)
collection_gaps.to_csv(QUALITY_DIR / "collection_gaps.csv", index=False)
missing_dates.to_csv(QUALITY_DIR / "missing_dates.csv", index=False)

print("Saved to:", QUALITY_DIR)

Saved to: C:\Users\f_tir\Documents\Code\Victoria-Urban-Planning\data\processed\cycling_counters\quality
